In [27]:
#| include: false

# celda de compatibilidad con ambiente local windows, amd rocm y torch

import sys
import warnings
from unittest.mock import MagicMock

import torch
import torch.distributed as dist

for _name in (
    "Store",
    "ProcessGroup",
    "PrefixStore",
    "TCPStore",
    "HashStore",
    "ReduceOp",
    "Work",
):
    if not hasattr(dist, _name):
        setattr(dist, _name, type(_name, (object,), {}))


def _stub_submodule(full_name, as_package=False):
    if full_name in sys.modules:
        return sys.modules[full_name]
    parts = full_name.split(".")
    parent_name = ".".join(parts[:-1])
    leaf = parts[-1]

    stub = MagicMock(name=full_name)
    stub.__spec__ = None
    if as_package:
        stub.__path__ = []  # hace que el import machinery lo trate como paquete
    sys.modules[full_name] = stub

    if parent_name in sys.modules:
        setattr(sys.modules[parent_name], leaf, stub)
    return stub


_stub_submodule("torch.distributed.fsdp", as_package=True)

for _nested in [
    "torch.distributed.fsdp.fully_sharded_data_parallel",
    "torch.distributed.fsdp._flat_param",
    "torch.distributed.fsdp._fully_shard",
    "torch.distributed.fsdp._fsdp_extensions",
]:
    _stub_submodule(_nested)

for _mod in [
    "torch._C._distributed_c10d",
    "torch.testing._internal.distributed.fake_pg",
    "torch.distributed._shard",
]:
    _stub_submodule(_mod)
    
warnings.filterwarnings("ignore", category=UserWarning)

print(
    "Shim activo: torch.distributed.fsdp (paquete + submódulos anidados) neutralizado."
)

Shim activo: torch.distributed.fsdp (paquete + submódulos anidados) neutralizado.


DPO fine-tune de Qwen2.5-0.5b para reducir verbosidad.

In [28]:
#| echo: false
#| eval: true

from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
from datasets import Dataset, load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

set_seed(10_000)

In [29]:
#| include: false

dataset_path: Path = Path('data') / 'DPO_dataset.json'
dataset: Dataset = load_dataset(
    "json",
    data_files=str(dataset_path),
    split="train",
)

In [30]:
#| include: false

def text_generation(model, q, max_new_tok=512) -> str:
    messages: list[dict[str, str]] = [{"role": "user", "content": q}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs, max_new_tokens=max_new_tok, **MODEL_PARAMS
    )

    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(
            model_inputs.input_ids, generated_ids, strict=False
        )
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

## Model config

In [31]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
device: Literal['cpu', 'cuda'] = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [32]:
MODEL_PARAMS: dict[str, float] = {
    "temperature": 0.7,
    "top_p": 0.9,
}

In [33]:
EVAL_PROMPTS: list[str] = [
    "Explicame que es la fotosintesis.",
    "Cuales son las ventajas de hacer ejercicio regularmente?",
    "Como funciona una red neuronal?",
]

## Base Eval

In [34]:
#| echo: false
#| eval: true

baseline_responses: dict[str, str] = {}
for p in EVAL_PROMPTS:
    resp: str = text_generation(model=model, q=p)
    baseline_responses[p] = resp
    print(f"PROMPT: {p}")
    print(f"RESPUESTA BASE ({len(resp.split())} palabras):\n{resp}\n")
    print("=" * 80)


PROMPT: Explicame que es la fotosintesis.
RESPUESTA BASE (308 palabras):
Fotosynthesis, también conocida como fotosíntesis, es un proceso biológico de la vida etérea en los organismos vivos. Este proceso se refiere a la producción de energía mediante la utilización del alimento o sustancias orgánicas para el crecimiento y desarrollo de las plantas, árboles y otros tipos de animales.

La fotosíntesis es muy importante para las comunidades alimentarias y la conservación de los recursos naturales. Aquí hay algunos puntos clave sobre la fotosíntesis:

1. Importancia: La fotosíntesis es fundamental para la sobrevivencia de las especies, ya que proporciona energía necesaria para su crecimiento y reproducción.

2. Distribución global: La fotosíntesis ocurre en todos los organismos vivos, desde los más pequeños hasta los más grandes, lo que significa que es una tarea universal.

3. Variabilidad: Las formas de fotosíntesis pueden variar significativamente entre diferentes tipos de flora y fauna

## Training

In [35]:
training_args = DPOConfig(
    output_dir=Path('checkpoints') / 'adapter',
    num_train_epochs=3,
    learning_rate=1e-06,
    beta=0.1,  # controla cuanto puede alejarse la policy del modelo de referencia
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,  # cap total sequence length
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,  # ahorra memoria de activaciones en GPU
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
#| echo: false
#| eval: true

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.691247
20,0.693572
30,0.692341
40,0.691034


## Training losses

In [ ]:
#| echo: false
#| eval: true

logs: list[dict[str, float]] = trainer.state.log_history

steps: list[float] = [log["step"] for log in logs if "loss" in log]
losses: list[float] = [log["loss"] for log in logs if "loss" in log]

plt.figure(figsize=(16, 4))
plt.grid(True)
plt.plot(steps, losses, label="train loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("DPO Training Loss")
plt.legend()
plt.show()

## Final Eval

In [ ]:
#| echo: false
#| eval: true

trainer.save_model(Path('checkpoints') / 'adapter')

finetuned_responses: dict[str, str] = {}
model.eval()
for p in EVAL_PROMPTS:
    resp = text_generation(model=model, q=p)
    finetuned_responses[p] = resp

print("COMPARACION ANTES / DESPUES DEL FINE-TUNING (DPO)\n")
for p in EVAL_PROMPTS:
    base = baseline_responses[p]
    tuned = finetuned_responses[p]
    print(f"PROMPT: {p}")
    print(f"  Palabras antes:   {len(base.split())}")
    print(f"  Palabras despues: {len(tuned.split())}")
    print(f"  --- Respuesta ANTES ---\n{base}\n")
    print(f"  --- Respuesta DESPUES ---\n{tuned}\n")
    print("=" * 80)